In [89]:
from __future__ import annotations

import numpy as np
import openml
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC

In [90]:
SEED=0

In [91]:
def split_train_val_test(x, y, train_ratio: float, test_ratio: float):
    total = train_ratio + test_ratio

    if total < 0.5 or total > 0.95:
        raise ValueError("train_ratio + test_ratio must be between 0.5 and 0.95")

    x_train, x_temp, y_train, y_temp = train_test_split(
        x,
        y,
        test_size=1 - train_ratio,
        random_state=SEED
    )

    # everything goes to validation if test_ratio == 0
    if test_ratio == 0:
        return x_train, None, x_temp, y_train, None, y_temp

    val_ratio = 1 - train_ratio - test_ratio

    x_val, x_test, y_val, y_test = train_test_split(
        x_temp,
        y_temp,
        test_size=test_ratio / (test_ratio + val_ratio),
        random_state=SEED
    )

    return x_train, x_test, x_val, y_train, y_test, y_val

In [92]:
def svm_accuracy(x_train: np.ndarray, y_train: np.ndarray,
                 x_val: np.ndarray, y_val: np.ndarray) -> float:

    assert x_train.ndim == 2
    assert y_train.ndim == 1

    clf = make_pipeline(
        StandardScaler(),
        LinearSVC(random_state=SEED, tol=1e-5)
    )

    print(x_train)
    print(y_train)

    clf.fit(x_train, y_train)

    y_pred = clf.predict(x_val)
    return float(accuracy_score(y_val, y_pred))

In [93]:
def svm_feature_fig(x_train: np.ndarray, y_train: np.ndarray):
    n_features = x_train.shape[1]

    le = LabelEncoder()
    y_enc = le.fit_transform(y_train)

    fig = go.Figure()

    for i in range(n_features):
        fig.add_trace(
            go.Scatter(
                x=x_train[:, i],
                y=y_enc,
                mode="markers",
                marker={"color": y_enc, "showscale": True},
                visible=(i == 0)
            )
        )

    steps = []
    for i in range(n_features):
        step = {
            "method": "update",
            "args": [{"visible": [False] * n_features}],
            "label": f"f{i}"
        }
        step["args"][0]["visible"][i] = True
        steps.append(step)

    fig.update_layout(sliders=[{"steps": steps}])

    return fig

In [ ]:
# Load the TabArena-v0.1 suite
suite = openml.study.get_suite(457)
assert suite.tasks is not None

print(openml.config._root_cache_directory)

for task_id in suite.tasks:
    task = openml.tasks.get_task(task_id)

    if task.task_type_id != openml.tasks.TaskType.SUPERVISED_CLASSIFICATION:
        continue

    dataset = task.get_dataset()

    x, y, categorical_mask, feature_names = dataset.get_data(
        dataset_format="dataframe",
        target=dataset.default_target_attribute
    )

    assert isinstance(x, pd.DataFrame)
    assert isinstance(y, pd.Series)

    x = x.to_numpy()
    y = y.to_numpy()

    x_train, x_test, x_val, y_train, y_test, y_val = split_train_val_test(
        x, y,
        train_ratio=0.7,
        test_ratio=0
    )

    print(f"{dataset.name} ({task_id = }):")

    acc = svm_accuracy(x_train, y_train, x_val, y_val)
    fig = svm_feature_fig(x_train, y_train)

    print(f"Accuracy: {acc*100:.2f}%")
    fig.show()